In [14]:
import json
from pathlib import Path

import pandas as pd

from sentence_transformers import SentenceTransformer

In [15]:
CURRENT_DIR = Path.cwd()

if CURRENT_DIR.name == "notebooks":
    PROJECT_ROOT = CURRENT_DIR.parent
else:
    PROJECT_ROOT = CURRENT_DIR


DATA_DIR = PROJECT_ROOT / "data"
OUTPUT_DIR = PROJECT_ROOT / "outputs"


train_poc = pd.read_csv(
    DATA_DIR / "train_poc.csv"
)

test_poc = pd.read_csv(
    DATA_DIR / "test_poc.csv"
)


stage_a_train = pd.read_csv(
    OUTPUT_DIR / "stage_a_train_predictions.csv"
)

stage_a_test = pd.read_csv(
    OUTPUT_DIR / "stage_a_test_predictions.csv"
)


print("Train records:", len(train_poc))
print("Test records:", len(test_poc))
print("Stage A train predictions:", len(stage_a_train))
print("Stage A test predictions:", len(stage_a_test))

Train records: 3322
Test records: 819
Stage A train predictions: 3322
Stage A test predictions: 819


In [17]:
# CSV stores prediction lists as text.
stage_a_train["stage_a_predictions"] = (
    stage_a_train["stage_a_predictions"].apply(
        json.loads
    )
)

stage_a_test["stage_a_predictions"] = (
    stage_a_test["stage_a_predictions"].apply(
        json.loads
    )
)

In [18]:
# validation cell
# Confirm that every POC record has a Stage A result.
if not train_poc["id"].isin(
    stage_a_train["id"]
).all():
    raise ValueError(
        "Some training records do not have Stage A predictions."
    )


if not test_poc["id"].isin(
    stage_a_test["id"]
).all():
    raise ValueError(
        "Some test records do not have Stage A predictions."
    )


print(
    "Stage A prediction IDs match the POC datasets."
)

Stage A prediction IDs match the POC datasets.


In [19]:
# Define the risk descriptions
RISK_TAXONOMY = {
    "weather_disruption": {
        "name": "Weather Disruption",
        "description": (
            "Disruption to trade, transport, ports, logistics, or infrastructure "
            "caused by severe weather such as storms, flooding, high winds, "
            "cyclones, or similar weather-related hazards."
        )
    },

    "natural_disaster": {
        "name": "Natural Disaster",
        "description": (
            "Disruption caused by natural disasters such as earthquakes, "
            "tsunamis, volcanic activity, landslides, or other geological hazards."
        )
    },

    "port_operational_disruption": {
        "name": "Port Operational Disruption",
        "description": (
            "Disruption to normal port or cargo operations caused by congestion, "
            "capacity limitations, operational delays, cargo disruption, "
            "or reduced terminal efficiency."
        )
    },

    "port_closure": {
        "name": "Port Closure",
        "description": (
            "Full or partial closure, suspension, or shutdown of a port, terminal, "
            "pier, berth, or related maritime facility."
        )
    },

    "labor_strike_disruption": {
        "name": "Labor / Strike Disruption",
        "description": (
            "Disruption caused by worker strikes, industrial action, labor disputes, "
            "walkouts, or related workforce actions affecting transport or logistics."
        )
    },

    "maritime_security_navigation_disruption": {
        "name": "Maritime Security / Navigation Disruption",
        "description": (
            "Disruption or increased operational risk to maritime transport caused "
            "by maritime advisories, piracy, security threats, navigation restrictions, "
            "or waterway closures and disruptions."
        )
    }
}

In [ ]:
import ast
import numpy as np
import torch

from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report
from sklearn.metrics import f1_score
from sklearn.metrics import precision_recall_fscore_support
from sklearn.model_selection import KFold
from sklearn.model_selection import cross_val_predict
from sklearn.multiclass import OneVsRestClassifier
from sklearn.preprocessing import MultiLabelBinarizer


# Restore the true risk lists loaded from CSV.
def restore_list(value):
    if isinstance(value, list):
        return value

    if pd.isna(value):
        return []

    return ast.literal_eval(value)


train_poc["true_risks"] = (
    train_poc["true_risks"].apply(
        restore_list
    )
)

test_poc["true_risks"] = (
    test_poc["true_risks"].apply(
        restore_list
    )
)


RISK_IDS = list(
    RISK_TAXONOMY.keys()
)


label_binarizer = MultiLabelBinarizer(
    classes=RISK_IDS
)

train_target_matrix = label_binarizer.fit_transform(
    train_poc["true_risks"]
)

Risk embeddings shape: (6, 384)


In [ ]:
# model-loading

# Stop execution if a Colab GPU is not enabled.
if not torch.cuda.is_available():
    raise RuntimeError(
        "GPU is not enabled. Select Runtime > Change runtime type > GPU."
    )


DEVICE = "cuda"

MODEL_NAME = (
    "sentence-transformers/all-MiniLM-L6-v2"
)


semantic_model = SentenceTransformer(
    MODEL_NAME,
    device=DEVICE
)


print("Device:", DEVICE)
print("Semantic model:", MODEL_NAME)

Batches:   0%|          | 0/104 [00:00<?, ?it/s]

Batches:   0%|          | 0/26 [00:00<?, ?it/s]

Train embeddings shape: (3322, 384)
Test embeddings shape: (819, 384)


In [ ]:
# Generate semantic embeddings
train_texts = (
    train_poc["input_text"]
    .fillna("")
    .astype(str)
    .tolist()
)

test_texts = (
    test_poc["input_text"]
    .fillna("")
    .astype(str)
    .tolist()
)


train_embeddings = semantic_model.encode(
    train_texts,
    batch_size=64,
    show_progress_bar=True,
    convert_to_numpy=True,
    normalize_embeddings=True
)

test_embeddings = semantic_model.encode(
    test_texts,
    batch_size=64,
    show_progress_bar=True,
    convert_to_numpy=True,
    normalize_embeddings=True
)


print(
    "Train embeddings shape:",
    train_embeddings.shape
)

print(
    "Test embeddings shape:",
    test_embeddings.shape
)

Train semantic scores shape: (3322, 6)
Test semantic scores shape: (819, 6)


In [ ]:
# Train the multilabel semantic classifier

# Logistic regression learns one classifier
# for each target risk.
base_classifier = LogisticRegression(
    max_iter=300,
    solver="liblinear",
    class_weight="balanced",
    random_state=42
)

stage_b_model = OneVsRestClassifier(
    base_classifier,
    n_jobs=-1
)


# Out-of-fold probabilities provide a fairer
# estimate for selecting the thresholds.
cross_validation = KFold(
    n_splits=3,
    shuffle=True,
    random_state=42
)

train_stage_b_probabilities = cross_val_predict(
    stage_b_model,
    train_embeddings,
    train_target_matrix,
    cv=cross_validation,
    method="predict_proba"
)


# Train the final model using all training records.
stage_b_model.fit(
    train_embeddings,
    train_target_matrix
)

test_stage_b_probabilities = (
    stage_b_model.predict_proba(
        test_embeddings
    )
)


print(
    "Training probabilities shape:",
    train_stage_b_probabilities.shape
)

print(
    "Test probabilities shape:",
    test_stage_b_probabilities.shape
)

In [ ]:
# Select a threshold for each risk
# Different risks need different probability thresholds.
threshold_options = np.arange(
    0.20,
    0.81,
    0.05
)

stage_b_thresholds = {}


for risk_index, risk_id in enumerate(RISK_IDS):

    best_threshold = 0.50
    best_f1 = 0.0

    for threshold in threshold_options:

        predicted_labels = (
            train_stage_b_probabilities[
                :,
                risk_index
            ] >= threshold
        ).astype(int)

        threshold_f1 = f1_score(
            train_target_matrix[
                :,
                risk_index
            ],
            predicted_labels,
            zero_division=0
        )

        if threshold_f1 > best_f1:
            best_f1 = threshold_f1
            best_threshold = threshold

    stage_b_thresholds[risk_id] = (
        float(best_threshold)
    )

    print(
        risk_id,
        "threshold:",
        round(best_threshold, 2),
        "F1:",
        round(best_f1, 4)
    )

Stage B training classifications: 3322
Stage B test classifications: 819


In [ ]:
# Convert probabilities into predictions
def get_stage_b_predictions(
    probability_matrix
):
    all_predictions = []

    for row_probabilities in probability_matrix:

        row_predictions = [
            risk_id
            for risk_index, risk_id in enumerate(
                RISK_IDS
            )
            if row_probabilities[risk_index]
            >= stage_b_thresholds[risk_id]
        ]

        all_predictions.append(
            row_predictions
        )

    return all_predictions

,id,Headline,stage_b_predictions
0,552,Severe winds caused brief suspension at Port o...,"[port_operational_disruption, port_closure]"
1,2973,USA: 'Fridays for Future' climate change prote...,[weather_disruption]
2,6,UPDATE - Indonesia: Severe winds damage infras...,[weather_disruption]
3,4537,UPDATE 1 - Refrigerated container import capac...,[port_operational_disruption]
4,3771,Marine wind warning issued for Port of Sydney ...,"[weather_disruption, maritime_security_navigat..."
5,3090,WATCH FOR: Road cargo and transport disruption...,[labor_strike_disruption]
6,1947,Osaka’s G20 summit likely to impede logistics ...,[maritime_security_navigation_disruption]
7,914,UPDATE: Up to 11 vessels waiting for a berth a...,[port_closure]
8,3927,Port congestion reported at Port of Dammam due...,"[port_operational_disruption, port_closure]"
9,809,"UPDATE - USA, Virginia: Tropical Storm Michael...",[weather_disruption]


In [ ]:
# Generate Stage B predictions
train_poc["stage_b_predictions"] = (
    get_stage_b_predictions(
        train_stage_b_probabilities
    )
)

test_poc["stage_b_predictions"] = (
    get_stage_b_predictions(
        test_stage_b_probabilities
    )
)


train_empty_predictions = (
    train_poc["stage_b_predictions"].apply(
        len
    ) == 0
).sum()

test_empty_predictions = (
    test_poc["stage_b_predictions"].apply(
        len
    ) == 0
).sum()


print(
    "Stage B training classifications:",
    len(train_poc)
)

print(
    "Stage B test classifications:",
    len(test_poc)
)

print(
    "Training records without a confident prediction:",
    train_empty_predictions
)

print(
    "Test records without a confident prediction:",
    test_empty_predictions
)

In [ ]:
# Inspect predictions
train_poc[
    [
        "id",
        "Headline",
        "true_risks",
        "stage_b_predictions"
    ]
].head(10)

In [ ]:
# Define the Stage B evaluation function
RISK_IDS = list(
    RISK_TAXONOMY.keys()
)


def evaluate_stage_b(dataframe, split_name):

    label_binarizer = MultiLabelBinarizer(
        classes=RISK_IDS
    )

    true_matrix = label_binarizer.fit_transform(
        dataframe["true_risks"]
    )

    predicted_matrix = label_binarizer.transform(
        dataframe["stage_b_predictions"]
    )

    print(
        f"{split_name} Stage B classification report"
    )

    print(
        classification_report(
            true_matrix,
            predicted_matrix,
            target_names=[
                RISK_TAXONOMY[risk_id]["name"]
                for risk_id in RISK_IDS
            ],
            zero_division=0
        )
    )

    micro_precision, micro_recall, micro_f1, _ = (
        precision_recall_fscore_support(
            true_matrix,
            predicted_matrix,
            average="micro",
            zero_division=0
        )
    )

    macro_precision, macro_recall, macro_f1, _ = (
        precision_recall_fscore_support(
            true_matrix,
            predicted_matrix,
            average="macro",
            zero_division=0
        )
    )

    exact_match = np.mean(
        [
            set(true_risks) == set(predicted_risks)
            for true_risks, predicted_risks in zip(
                dataframe["true_risks"],
                dataframe["stage_b_predictions"]
            )
        ]
    )

    print(
        "Micro precision:",
        round(micro_precision, 4)
    )

    print(
        "Micro recall:",
        round(micro_recall, 4)
    )

    print(
        "Micro F1:",
        round(micro_f1, 4)
    )

    print(
        "Macro precision:",
        round(macro_precision, 4)
    )

    print(
        "Macro recall:",
        round(macro_recall, 4)
    )

    print(
        "Macro F1:",
        round(macro_f1, 4)
    )

    print(
        "Exact multi-label match:",
        round(exact_match, 4)
    )

    return {
        "micro_precision": micro_precision,
        "micro_recall": micro_recall,
        "micro_f1": micro_f1,
        "macro_precision": macro_precision,
        "macro_recall": macro_recall,
        "macro_f1": macro_f1,
        "exact_match": exact_match
    }

In [ ]:
# Evaluate training predictions
train_stage_b_metrics = evaluate_stage_b(
    train_poc,
    "Training"
)


Training Stage B classification report
                                           precision    recall  f1-score   support

                       Weather Disruption       0.92      0.53      0.68      1300
                         Natural Disaster       0.64      0.87      0.74       103
              Port Operational Disruption       0.66      0.69      0.67      1394
                             Port Closure       0.27      0.82      0.40       355
                Labor / Strike Disruption       0.96      0.75      0.84       740
Maritime Security / Navigation Disruption       0.31      0.39      0.34       464

                                micro avg       0.60      0.63      0.62      4356
                                macro avg       0.63      0.67      0.61      4356
                             weighted avg       0.72      0.63      0.65      4356
                              samples avg       0.65      0.67      0.63      4356

Micro precision: 0.6029
Micro recall: 0.6348


In [ ]:
# Evaluate test predictions

test_stage_b_metrics = evaluate_stage_b(
    test_poc,
    "Test"
)

Test Stage B classification report
                                           precision    recall  f1-score   support

                       Weather Disruption       0.94      0.53      0.68       341
                         Natural Disaster       0.69      0.89      0.78        35
              Port Operational Disruption       0.68      0.68      0.68       351
                             Port Closure       0.27      0.82      0.41        85
                Labor / Strike Disruption       0.95      0.79      0.86       179
Maritime Security / Navigation Disruption       0.30      0.43      0.35        97

                                micro avg       0.62      0.65      0.63      1088
                                macro avg       0.64      0.69      0.63      1088
                             weighted avg       0.74      0.65      0.66      1088
                              samples avg       0.66      0.68      0.65      1088

Micro precision: 0.6167
Micro recall: 0.6461
Micr

In [32]:
# Compare the two reports
stage_b_metric_comparison = pd.DataFrame(
    [
        train_stage_b_metrics,
        test_stage_b_metrics
    ],
    index=[
        "Training",
        "Test"
    ]
)

stage_b_metric_comparison.round(4)

,micro_precision,micro_recall,micro_f1,macro_precision,macro_recall,macro_f1,exact_match
Training,0.6029,0.6348,0.6184,0.6263,0.6748,0.6126,0.4266
Test,0.6167,0.6461,0.6311,0.6359,0.6903,0.6250,0.4396


In [ ]:
# Merge the Stage A and Stage B results using id.
train_results = train_poc[
    [
        "id",
        "true_risks",
        "stage_b_predictions"
    ]
].merge(
    stage_a_train[
        [
            "id",
            "stage_a_predictions"
        ]
    ],
    on="id",
    how="left",
    validate="one_to_one"
)


test_results = test_poc[
    [
        "id",
        "true_risks",
        "stage_b_predictions"
    ]
].merge(
    stage_a_test[
        [
            "id",
            "stage_a_predictions"
        ]
    ],
    on="id",
    how="left",
    validate="one_to_one"
)


print(
    "Merged training records:",
    len(train_results)
)

print(
    "Merged test records:",
    len(test_results)
)

In [ ]:
# Decide whether each record should be sent to Stage C.
def get_stage_c_decision(row):

    stage_a_predictions = (
        row["stage_a_predictions"]
    )

    stage_b_predictions = (
        row["stage_b_predictions"]
    )

    stage_a_set = set(
        stage_a_predictions
    )

    stage_b_set = set(
        stage_b_predictions
    )


    # Accept an exact, non-empty agreement.
    if (
        stage_a_set
        and stage_a_set == stage_b_set
    ):
        return pd.Series({
            "route_to_stage_c": False,
            "accepted_predictions": stage_a_predictions,
            "stage_c_route_reason": (
                "stage_a_stage_b_agree"
            )
        })


    # Use Stage B when Stage A has no prediction.
    if (
        not stage_a_set
        and stage_b_set
    ):
        return pd.Series({
            "route_to_stage_c": False,
            "accepted_predictions": stage_b_predictions,
            "stage_c_route_reason": (
                "stage_a_empty_stage_b_confident"
            )
        })


    # Route every other case to Stage C.
    if (
        not stage_a_set
        and not stage_b_set
    ):
        route_reason = "both_stages_empty"

    elif (
        stage_a_set
        and not stage_b_set
    ):
        route_reason = "stage_b_empty"

    else:
        route_reason = "predictions_disagree"


    return pd.Series({
        "route_to_stage_c": True,
        "accepted_predictions": [],
        "stage_c_route_reason": route_reason
    })


train_routing = train_results.apply(
    get_stage_c_decision,
    axis=1
)

test_routing = test_results.apply(
    get_stage_c_decision,
    axis=1
)


train_results = pd.concat(
    [
        train_results,
        train_routing
    ],
    axis=1
)

test_results = pd.concat(
    [
        test_results,
        test_routing
    ],
    axis=1
)

In [ ]:
# Compare Stage A and Stage B before Stage C.
def compare_prediction_methods(dataframe):

    true_matrix = label_binarizer.transform(
        dataframe["true_risks"]
    )

    comparison_rows = []


    prediction_columns = {
        "Stage A": "stage_a_predictions",
        "Stage B": "stage_b_predictions"
    }


    for method_name, column_name in (
        prediction_columns.items()
    ):

        predicted_matrix = label_binarizer.transform(
            dataframe[column_name]
        )

        micro_precision, micro_recall, micro_f1, _ = (
            precision_recall_fscore_support(
                true_matrix,
                predicted_matrix,
                average="micro",
                zero_division=0
            )
        )

        _, _, macro_f1, _ = (
            precision_recall_fscore_support(
                true_matrix,
                predicted_matrix,
                average="macro",
                zero_division=0
            )
        )

        exact_match = np.mean(
            np.all(
                true_matrix == predicted_matrix,
                axis=1
            )
        )

        coverage = dataframe[column_name].apply(
            lambda predictions: len(predictions) > 0
        ).mean()

        comparison_rows.append({
            "method": method_name,
            "micro_precision": micro_precision,
            "micro_recall": micro_recall,
            "micro_f1": micro_f1,
            "macro_f1": macro_f1,
            "exact_match": exact_match,
            "coverage": coverage
        })


    return pd.DataFrame(
        comparison_rows
    ).set_index(
        "method"
    )

In [ ]:
# Training comparison
train_method_comparison = (
    compare_prediction_methods(
        train_results
    )
)

train_method_comparison.round(4)

In [ ]:
# Test comparison
test_method_comparison = (
    compare_prediction_methods(
        test_results
    )
)

test_method_comparison.round(4)

In [ ]:
# Check how many records will be processed by Stage C.
print(
    "Training records routed to Stage C:",
    int(train_results["route_to_stage_c"].sum()),
    "of",
    len(train_results)
)

print(
    train_results[
        "stage_c_route_reason"
    ].value_counts()
)


print(
    "\nTest records routed to Stage C:",
    int(test_results["route_to_stage_c"].sum()),
    "of",
    len(test_results)
)

print(
    test_results[
        "stage_c_route_reason"
    ].value_counts()
)

In [ ]:
# Prepare Stage B probabilities for export
train_probability_records = [
    {
        risk_id: round(
            float(row_probabilities[risk_index]),
            6
        )
        for risk_index, risk_id in enumerate(
            RISK_IDS
        )
    }
    for row_probabilities in (
        train_stage_b_probabilities
    )
]


test_probability_records = [
    {
        risk_id: round(
            float(row_probabilities[risk_index]),
            6
        )
        for risk_index, risk_id in enumerate(
            RISK_IDS
        )
    }
    for row_probabilities in (
        test_stage_b_probabilities
    )
]

In [ ]:
# Export Stage B predictions
OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True
)


stage_b_train_output = pd.DataFrame({
    "id": train_poc["id"],
    "stage_b_predictions": (
        train_poc["stage_b_predictions"].apply(
            lambda predictions: json.dumps(
                predictions,
                ensure_ascii=False
            )
        )
    ),
    "stage_b_probabilities": [
        json.dumps(
            probabilities,
            ensure_ascii=False
        )
        for probabilities in (
            train_probability_records
        )
    ]
})


stage_b_test_output = pd.DataFrame({
    "id": test_poc["id"],
    "stage_b_predictions": (
        test_poc["stage_b_predictions"].apply(
            lambda predictions: json.dumps(
                predictions,
                ensure_ascii=False
            )
        )
    ),
    "stage_b_probabilities": [
        json.dumps(
            probabilities,
            ensure_ascii=False
        )
        for probabilities in (
            test_probability_records
        )
    ]
})


stage_b_train_output.to_csv(
    OUTPUT_DIR / "stage_b_train_predictions.csv",
    index=False
)

stage_b_test_output.to_csv(
    OUTPUT_DIR / "stage_b_test_predictions.csv",
    index=False
)


print(
    "Saved:",
    OUTPUT_DIR / "stage_b_train_predictions.csv"
)

print(
    "Saved:",
    OUTPUT_DIR / "stage_b_test_predictions.csv"
)

In [ ]:
# Prepare probability records with their matching IDs.
train_probability_data = pd.DataFrame({
    "id": train_poc["id"],
    "stage_b_probabilities": (
        train_probability_records
    )
})

test_probability_data = pd.DataFrame({
    "id": test_poc["id"],
    "stage_b_probabilities": (
        test_probability_records
    )
})


# Create the complete Stage C input data.
train_stage_c_input = train_poc[
    [
        "id",
        "input_text"
    ]
].merge(
    train_results[
        [
            "id",
            "stage_a_predictions",
            "stage_b_predictions",
            "route_to_stage_c",
            "accepted_predictions",
            "stage_c_route_reason"
        ]
    ],
    on="id",
    how="left",
    validate="one_to_one"
).merge(
    train_probability_data,
    on="id",
    how="left",
    validate="one_to_one"
)


test_stage_c_input = test_poc[
    [
        "id",
        "input_text"
    ]
].merge(
    test_results[
        [
            "id",
            "stage_a_predictions",
            "stage_b_predictions",
            "route_to_stage_c",
            "accepted_predictions",
            "stage_c_route_reason"
        ]
    ],
    on="id",
    how="left",
    validate="one_to_one"
).merge(
    test_probability_data,
    on="id",
    how="left",
    validate="one_to_one"
)